In [ ]:
# install music21 lib - pip install music21

import numpy as np
from music21 import converter, chord, instrument
import tensorflow as tf


In [ ]:
import glob
midi_file=glob.glob("/content/*.mid")
print("total number of ",len(midi_file))
print("midi files",midi_file[:5])
notes=[]
for file in midi_file:
  try:
    midi=converter.parse(file)
    parts=instrument.partitionByInstrument(midi)
    if parts:
      notes_to_parse=parts.parts[0].recurse()
    else:
      notes_to_parse=midi.flat.notes
    for element in notes_to_parse:
      if isinstance(element,note.Note):
        notes.append(str(element.pitch))
      elif isinstance(element,chord.Chord):
        notes.append(''.join(str(n) for n in element.normalOrder))
  except Exception as e:
    print("error reading {file}: {e}")
print("total notes:",len(notes))
print(notes[:20])
pitchnames=sorted(set(notes))
print("total unique notes",len(pitchnames))
note_to_int={note:number for number,note in enumerate(pitchnames)}
print("Vocabulary sizes:",len(note_to_int))


In [ ]:
sequence_length=100
network_input=[]
network_output=[]
for i in range(len(notes)-sequence_length):
  sequence_in=notes[i:i +sequence_length]
  sequence_out=notes[i + sequence_length]
  network_input.append([note_to_int[n] for n in sequence_in])
  network_output.append(note_to_int[sequence_out])

print("input sequence",len(network_input))
print("output sequence",len(network_output))

In [ ]:
n_patterns=len(network_input)
network_input=np.reshape(
    network_input,
    (n_patterns,sequence_length,1)
)
network_input=network_input/float(len(pitchnames))
print(network_input.shape)


In [ ]:
network_output = tf.keras.utils.to_categorical(network_output)
print(network_output.shape)
## starting the core model cnn
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dropout,Dense
model=Sequential()
model.add(LSTM(
    512,
    input_shape=(network_input.shape[1],network_input.shape[2]),
    return_sequences=True
))
model.add(Dropout(0.3))
model.add(LSTM(512))
model.add(Dense(250,activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(network_output.shape[1],activation='softmax'))
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="loss",
    verbose=1,
    save_best_only=True,
    mode="min"
)
history=model.fit(
    network_input,
    network_output,
    epochs=5,
    batch_size=64,
    callbacks=[checkpoint]
)


In [ ]:
from tensorflow.keras.models import load_model
import random
load_model("best_model.keras")
start=random.randint(0,len(network_input)-1)
int_to_note={number: note for number, note in enumerate(pitchnames)}
pattern = network_input[start]
prediction_output=[]

for _ in range(500):
  prediction_input=np.reshape(pattern,(1,len(pattern),1))
  prediction=model.predict(prediction_input,verbose=1)
  index= np.argmax(prediction)
  result=int_to_note[index]
  prediction_output.append(result)
  np.append(pattern,index/float(len(pitchnames)))
  pattern=pattern[1:]


In [ ]:
from music21 import stream, note, chord, instrument

offset = 0
output_notes = []

for pattern in prediction_output:

    if "." in pattern or pattern.isdigit():
        notes_in_chord = pattern.split(".")
        chord_notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            chord_notes.append(new_note)

        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)

    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("MIDI file saved as generated_music.mid")

In [ ]:
# to play the music and giving some time interval to stop the intervention of audio

import pygame
import time


pygame.init()
pygame.mixer.init()

pygame.mixer.music.load("/content/generated_music.mid")
pygame.mixer.music.play()

print("Playing generated_music.mid...")

"""
while pygame.mixer.music.get_busy():
    time.sleep(1)
"""

time.sleep(10)   # Play for 10 seconds

pygame.mixer.music.stop()
pygame.quit()

print("Music playback finished.")

In [ ]:
# Install dependencies
!apt-get -qq update
!apt-get -qq install -y fluidsynth
!pip -q install midi2audio

# Download a SoundFont
!wget -q -O FluidR3_GM.sf2 https://github.com/urish/cinto/raw/master/media/FluidR3_GM.sf2

# Convert MIDI to WAV
from midi2audio import FluidSynth
from IPython.display import Audio, display

midi_file = "/content/generated_music.mid"
wav_file = "/content/generated_music.wav"

fs = FluidSynth(sound_font="FluidR3_GM.sf2")
fs.midi_to_audio(midi_file, wav_file)

print("Conversion completed successfully!")

# Play the generated audio
display(Audio(wav_file, autoplay=True))